In [13]:
import numpy as np
import jax

def f(x):  # function we're benchmarking (works in both NumPy & JAX)
    return x @ x

In [21]:
x_np = np.ones((4096, 4096), dtype=np.float32)  # same as JAX default dtype
%timeit f(x_np)  # measure NumPy runtime

349 ms ± 13 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
# measure JAX device transfer time
%time x_jax = jax.device_put(x_np).block_until_ready()

CPU times: user 3.84 ms, sys: 136 ms, total: 140 ms
Wall time: 35.8 ms


In [23]:
f_jit = jax.jit(f)
%time f_jit(x_jax).block_until_ready()  # measure JAX compilation time
%timeit f_jit(x_jax).block_until_ready()  # measure JAX runtime

CPU times: user 3.75 s, sys: 134 ms, total: 3.88 s
Wall time: 1.85 s
1.18 ms ± 2.55 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [24]:
x_np = np.ones((4096, 4096), dtype=jax.numpy.bfloat16)  # same as JAX default dtype
%timeit f(x_np)  # measure NumPy runtime

357 ms ± 687 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [25]:
# measure JAX device transfer time
%time x_jax = jax.device_put(x_np).block_until_ready()

CPU times: user 2.36 ms, sys: 11.1 ms, total: 13.5 ms
Wall time: 3.79 ms


In [26]:
f_jit = jax.jit(f)
%time f_jit(x_jax).block_until_ready()  # measure JAX compilation time
%timeit f_jit(x_jax).block_until_ready()  # measure JAX runtime

CPU times: user 2.67 s, sys: 233 ms, total: 2.9 s
Wall time: 1.76 s
666 μs ± 2.06 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [27]:
x_np.dtype

dtype(bfloat16)

In [5]:
import torch
import time

device = 'cuda'
dtype = torch.bfloat16  # or torch.bfloat16

A = torch.randn(4096, 4096, dtype=dtype, device=device)
B = torch.randn(4096, 4096, dtype=dtype, device=device)

# Warmup
for _ in range(10):
    C = torch.matmul(A, B)
torch.cuda.synchronize()

# Benchmark
times = []
for _ in range(100):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    C = torch.matmul(A, B)
    end.record()
    
    torch.cuda.synchronize()
    times.append(start.elapsed_time(end))  # milliseconds

print(f"Mean: {sum(times)/len(times):.3f} ms")

Mean: 0.548 ms
